# Transformations using spark DF

### Creating dataset

In [0]:
# All imports
from pyspark.sql.functions import *

In [0]:
data = [
    (1, "Shubham", "IT", 70000, "2022-01-10"),
    (2, "Amit", "IT", 80000, "2021-03-15"),
    (3, "Neha", "HR", 60000, "2020-07-01"),
    (4, "Rahul", "HR", 65000, "2021-09-20"),
    (5, "Priya", "Finance", 90000, "2019-11-11"),
    (6, "Ankit", "Finance", 85000, "2022-06-01")
]

columns = ["emp_id", "name", "department", "salary", "join_date"]

df = spark.createDataFrame(data, columns) \
          .withColumn("join_date", to_date("join_date"))

df.show()
df.printSchema()

### Step 1: SELECT & COLUMN OPERATIONS

In [0]:
# 1.1 Select Columns
df.select("name","salary")

# 1.2 Select with Expression
df.select(
    "name",
    (col("salary") + 5000).alias("revised_salary")
)

# Select All Columns
df.select("*")

### Step 2: FILTER / WHERE

In [0]:
# filter rows
df.filter(col('salary') > 70000)

# Multiple Conditions
df.filter((col('salary') > 70000) & (col('department') == 'IT'))

# Where is an alias for filter
df.where(col("department") == "HR").show()  # df.where("department = 'HR'")

### Step 3: WITHCOLUMN (Transformations)

In [0]:
# 3.1 Add New Column
df.withColumn("salary_in_lakh", col("salary") / 100000)

# 3.2 Update Existing Column
df.withColumn("salary", col("salary") * 1.1)

# 3.2.1 Update Existing Multiple Columns
df.withColumns({
  "salary" : col("salary")*1.1,
  "bonus" : col("salary")*0.2
})

# 3.3 Conditional Column (CASE WHEN)
df.withColumn(
  "grade",
  when(col("salary") >= 85000, "A")
    .when(col("salary") >= 70000, "B")
    .otherwise("C")
)

### Step 4: SORT / ORDER BY

In [0]:
df.orderBy(col("salary").desc())   # df.sort("salary").show()
df.sort("department", col("salary").desc())

### Step 5: DISTINCT & DROP DUPLICATES

In [0]:
df.select("department").distinct().show()

In [0]:
df.dropDuplicates(["department"]).show()   #dropDuplicates need list of column values

### Step 6: AGGREGATIONS

In [0]:
# 6.1 Basic Aggregations
df.agg(
  avg("salary").alias("average_salary"),
  max("salary").alias("max_salary"),
  min("salary").alias("min_salary"),
  sum("salary").alias("total_salary"),
  count("*").alias("total_employees")
).show()

### Step 7: GROUP BY

In [0]:
df.groupBy("department").agg(
  avg("salary").alias("average_salary"),
  max("salary").alias("max_salary"),
  min("salary").alias("min_salary"),
  sum("salary").alias("total_salary"),
  count("*").alias("total_employees")
).show()

### HAVING equivalent

In [0]:
df.groupBy("department")\
    .agg(avg("salary").alias("avg_salary")) \
    .filter(col("avg_salary") > 70000)

### Step 8: JOINS (VERY IMPORTANT)

In [0]:
# create Department Table
dept_data = [
    ("IT", "Bangalore"),
    ("HR", "Mumbai"),
    ("Finance", "Delhi")
]

dept_df = spark.createDataFrame(dept_data, ["department","city"])

In [0]:
# Inner Join
df.join(dept_df, "department", "inner")

# Left Join
df.join(dept_df, "department", "left")

# Right Join
df.join(dept_df, "department", "right")

# Full Join
df.join(dept_df, "department", "outer") # here outer can be replaced with full

# Self Join
df.alias("e1").join(df.alias("e2"),
        col("e1.department") == col("e2.department"),
        "inner"
 ).select(col("e1.name").alias("emp"),col("e2.name").alias("manager"))

### Step 9: WINDOW FUNCTIONS (CRITICAL)

In [0]:
# Define Window
from pyspark.sql.window import Window
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())

# 9.1 ROW_NUMBER
df.withColumn("row_number", row_number().over(window_spec))

# 9.2 RANK
df.withColumn("rank", rank().over(window_spec))

# 9.3 DENSE_RANK
df.withColumn("dense_rank", dense_rank().over(window_spec))

### Step 10: LEAD & LAG

In [0]:
df.withColumn("previous_salary", lag("salary", 1).over(window_spec))

df.withColumn("next_salary", lead("salary", 1).over(window_spec))

### Step 11: RUNNING TOTAL & MOVING AVG

In [0]:
# definding Running window spec
running_window = Window.partitionBy("department").orderBy(col("join_date"))\
    .rowsBetween(Window.unboundedPreceding, Window.currentRow) # if we do no ahve preference to check all the previos records then rowsBeween is no need to mention

df.withColumn("running_salary", sum("salary").over(running_window))

### Step 12: PIVOT

In [0]:
display(df)
df.groupBy("department")\
    .pivot("name")\
    .avg("salary").display()

### Step 13: UNION, UNION ALL, UnionByName

In [0]:
df1 = df.filter(col("department") == "IT")
df2 = df.filter(col("department") == "HR")

df1.union(df2)        # There is no functional difference between union and unionAll in PySpark.
df1.unionByName(df2)  # Name-based UNION (Best practice):-1) Matches columns by column name, 2) Column order does NOT matter

### Step 14: NULL HANDLING

In [0]:
from pyspark.sql.functions import to_date, min, mean,median
from pyspark.sql.functions import *
data = [
    (1, "Shubham", "IT", 70000, "2022-01-10"),
    (2, "Amit", "IT", 80000, "2021-03-15"),
    (3, "Neha", "HR", 60000, "2020-07-01"),
    (4, "Rahul", "HR", 65000, "2021-09-20"),
    (5, "Priya", "Finance", 90000, "2019-11-11"),
    (6, "Ankit", "Finance", 85000, "2022-06-01"),
    (7, "Ishani", "IT", None, "2021-04-01")
]

columns = ["emp_id", "name", "department", "salary", "join_date"]

df_null = spark.createDataFrame(data, columns) \
          .withColumn("join_date", to_date("join_date"))

# fill null values with minimum salary
min_sal = df_null.select(min("salary")).collect()[0][0]
df_null.na.fill({"salary": min_sal})

# fill null values with mean
mean_sal = df_null.select(mean("salary")).collect()[0][0]
df_null.na.fill({"salary": mean_sal})

# fill null values with median
median_sal = df_null.select(median("salary")).collect()[0][0]
df_null.na.fill({"salary": median_sal})

# dropping null values
df_null.na.drop()

### Step 15: STRING FUNCTIONS

In [0]:
df.select(upper("name"), lower("department"), length("name"))

### Step 16: DATE FUNCTIONS

In [0]:
df.select(
    "name",
    year("join_date").alias("join_year"),
    month("join_date").alias("join_month"),
    dayofmonth("join_date").alias("join_day"),
    months_between(current_timestamp(), "join_date").alias("months_since_join")
).show()

### Step 17: REPARTITION & COALESCE

In [0]:
df.repartition(4)    # it will create 4 partition 
df.repartition("department") # if partition needed on sme column then we can use, specifically it can be used for joining operation(if department is the join condition then we can do partition on department)
df.repartition(4, "department")  # it will create 4 partition as per the department

In [0]:
df.coalesce(4)  # coalesce only ake one argument which is number of partition needed, it do not garenty uniform volumne of partition alike repartition

### 21. EXPLODE / POSEXPLODE / EXPLODE_OUTER

In [0]:
data = [
    (1, "Shubham", ["Spark", "Python", "SQL"]),
    (2, "Amit", ["Java", "Spring"]),
    (3, "Neha", None)
]

df_skill = spark.createDataFrame(data, ["emp_id", "name", "skills"])


**21.1 explode() – Converts array → rows**
remove records which having null recods in explode

In [0]:
df_skill.select(
    "emp_id",
    "name",
    explode("skills").alias("skill")
).display()

**21.2 explode_outer() – Keeps NULL rows
df_skill**

In [0]:
df_skill.select(
    "emp_id",
    "name",
    explode_outer("skills").alias("skill")
).display()

**21.3 posexplode() – Position + value**

In [0]:
df_skill.select("emp_id","name", posexplode("skills")).display()  # if wanted to include null values also then use posexplode_outer

**21.4 Explode Map Type**

In [0]:
map_data = [
    (1, {"Jan": 1000, "Feb": 2000}),
    (2, {"Jan": 1500})
]

df_map = spark.createDataFrame(map_data, ["emp_id", "sales"])

df_map.select(
    "emp_id",
    explode("sales").alias("month", "amount")
).show()


### 22. fillna() / dropna() / replace()

In [0]:
# Sample Data with NULLs
data = [
  (1,"IT",70000),
  (2,None,70000),
  (3,"HR",None),
  (4,None,None)
]

df_null = spark.createDataFrame(data,["emp_id","dept","salary"])

**22.1 fillna() – Single Value**

In [0]:
df_null.fillna("unknown").display() # here the data type of column consider the major role, as demaprtment column has string datatype that why only department value get filled but not salary as salary column has long dataType

**22.2 fillna() – Column-wise**

In [0]:
df_null.fillna({
    "department" :"unknown",
    "salary": 0                 # can calculate min/max/mean/median/mode salary from df
})

**22.3 Conditional NULL Handling (Recommended)**

In [0]:
df_null.withColumn(
  "salary", when(
                col("salary").isNull(), avg("salary").over(Window.partitionBy()))
                  .otherwise(col("salary"))).show()


**22.4 dropna()**

In [0]:
df_null.dropna() # drop rows / records where the null values are availble irrespective of column
df_null.dropna(subset=["salary"])  # drop rows / records where the null values are availble

### 23. MEAN, MAX, MEDIAN (Very Important)

**23.1 Mean / Avg, Min, Max**

In [0]:
df.agg(
  mean("salary").alias("avg_salary"),
  avg("salary").alias("avg_salary"),
  max("salary").alias("max_salary"),
  min("salary").alias("min_salary")
)

**23.3 MEDIAN (No direct function ❗)**

In [0]:
# median on entire column 
df.select(
  expr("percentile_approx(salary, 0.5)").alias("median_salary")
)

# Median Per Group
df.groupBy("department").agg(
  expr("percentile_approx(salary, 0.5)").alias("median_salary")
)

### 24. READ CSV – HANDLE CORRUPT RECORDS (VERY IMPORTANT)
This is Bronze layer / ingestion

In [0]:
# Define schema (Mandatory)
from pyspark.sql.functions import *

from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("emp_id", IntegerType()),
    StructField("name", StringType()),
    StructField("salary", IntegerType())
])

**24.2 Read CSV with Corrupt Record Capture**

In [0]:
df_csv = spark.read\
    .format("csv")\
    .schema(schema)\
    .option("header", "true")\
    .option("mode","PERMISSIVE")\
    .option("columnNameOfCorruptRecord", "_corrupt_record")\
    .load("/Volumes/shubham/sql_db/sample_data/employee.csv")

# df_csv.filter(col("_corrupt_record").isNotNUll()).show(truncate=False)
df_csv.display()

**24.3 CSV Read Modes (Interview MUST)**
| Mode            | Behavior                          |
| --------------- | --------------------------------- |
| `PERMISSIVE`    | Default, sets bad records to NULL |
| `DROPMALFORMED` | Drops bad rows                    |
| `FAILFAST`      | Job fails immediately             |

.option("mode","DROPMALFORMED")

### 25. IMPORTANT CSV READ PARAMETERS

In [0]:
spark.read \
  .option("header", "true") \
  .option("delimiter", ",") \
  .option("quote", "\"") \
  .option("escape", "\\") \
  .option("multiLine", "true") \
  .option("ignoreLeadingWhiteSpace", "true") \
  .option("ignoreTrailingWhiteSpace", "true") \
  .option("nullValue", "NULL") \
  .option("emptyValue", "") \
  .option("dateFormat", "yyyy-MM-dd") \
  .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
  .csv("/path/input/file.csv")

### 26. READ TABLE (DELTA / HIVE)

In [0]:
spark.read.table("bronze.employee")

### 27. PRODUCTION INGESTION BEST PRACTICE (Databricks)
**✔️ Bronze Layer**
- Schema enforcement
- Capture _corrupt_record
- No transformations

**✔️ Silver Layer**
- fillna
- Type casting
- Deduplication

**✔️ Gold Layer**
- Aggregations
- Window functions
- Business metrics

## Work count problem using Dataframe

In [0]:
from pyspark.sql.functions import col, split, explode, count

# Sample data, we can read file also
data = [("Spark is fast and Spark is scalable",)]
df = spark.createDataFrame(data, ["text"])

#Spliting the Column into list of words
df_split = df.withColumn("words", split(col("text"), " "))

# Exploding the list to get number of words present in list
df_explode = df_split.withColumn("word", explode(col("words"))).select(col("word"))
# df_explode.display()
df_explode.groupby(col("word")).count().alias("count").display()